In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from tqdm import tqdm

In [2]:
df = pd.read_csv("data/ptb-xl/final_dataset.csv")

print("Dataset Shape :", df.shape)

display(df.head())

Dataset Shape : (19492, 17)


,ecg_id,heart_rate,mean_rr,std_rr,min_rr,max_rr,rmssd,nn50,pnn50,pr_mean,qrs_mean,qt_mean,qtc_mean,r_amp_mean,t_amp_mean,st_mean,label
0,1,63.844881,939.777778,16.691833,912.0,964.0,23.291629,0,0.000000,127.400000,99.666667,394.222222,406.657247,4.009720,2.433324,0.008104,NORM
1,2,47.738122,1256.857143,78.273200,1162.0,1370.0,54.546616,3,0.500000,164.000000,98.000000,400.857143,357.558139,4.527706,2.282062,0.131958,NORM
2,3,63.799622,940.444444,18.397531,916.0,974.0,15.524175,0,0.000000,119.750000,101.000000,429.200000,442.633681,4.561276,1.320380,0.191043,NORM
3,4,75.015628,799.833333,40.410051,732.0,872.0,36.271702,2,0.181818,140.444444,94.181818,361.818182,404.482894,3.935648,1.821734,0.243649,NORM
4,5,66.283694,905.200000,48.639079,830.0,984.0,57.669557,5,0.555556,139.800000,102.571429,376.545455,395.772065,4.913701,1.918733,0.110245,NORM


In [3]:
# Features
X = df.drop(columns=["label"])

# Labels
y = df["label"]

print("Feature shape :", X.shape)
print("Number of labels :", y.nunique())

Feature shape : (19492, 16)
Number of labels : 51


In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Keep the original labels if you need them later
y_original = y.copy()

# Encode labels to integers
y_encoded = label_encoder.fit_transform(y)

print("Number of classes:", len(label_encoder.classes_))
print("First 10 encoded labels:", y_encoded[:10])

Number of classes: 51
First 10 encoded labels: [38 38 38 38 38 38 38 14 38 38]


In [5]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

print(X.shape)

(19492, 16)


In [6]:
y = df["label"]
print("Number of unique labels:", y.nunique())

label_counts = y.value_counts()

display(label_counts)

print("\nLabels having fewer than 5 samples:\n")
display(label_counts[label_counts < 5])

Number of unique labels: 51


label
NORM       8808
IMI        1486
NDT        1477
ASMI       1252
LVH         877
LAFB        846
IRBBB       782
NST_        436
CRBBB       345
IVCD        287
CLBBB       280
ILMI        263
ISCAL       245
ISC_        243
PVC         225
PACE        199
1AVB        164
AMI         155
INJAS       128
ALMI        118
ISCAS       106
ISCIN        80
LNGQT        71
LAO/LAE      69
LMI          59
ISCIL        55
WPW          52
AFIB         44
ISCLA        38
IPLMI        38
INJAL        35
ILBBB        29
EL           28
RAO/RAE      26
IPMI         24
AFLT         23
LPFB         21
ISCAN        18
DIG          13
PMI           8
SEHYP         8
RVH           7
SR            5
INJIL         3
STACH         3
3AVB          3
INJLA         3
2AVB          3
PSVT          2
INJIN         1
PAC           1
Name: count, dtype: int64


Labels having fewer than 5 samples:



label
INJIL    3
STACH    3
3AVB     3
INJLA    3
2AVB     3
PSVT     2
INJIN    1
PAC      1
Name: count, dtype: int64

In [7]:
X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Training:", len(X_train))
print("Testing :", len(X_test))

Training: 15593
Testing : 3899


In [8]:
print(type(y_train_encoded))
print(y_train_encoded[:10])

<class 'numpy.ndarray'>
[38 38  8  7 50 14 38 31 14 31]


In [9]:
import torch
from torch.utils.data import TensorDataset

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print("Train dataset:", len(train_dataset))
print("Test dataset :", len(test_dataset))

Train dataset: 15593
Test dataset : 3899


In [10]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 128

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Test batches :", len(test_loader))

Train batches: 122
Test batches : 31


In [11]:
import torch
import torch.nn as nn


# ============================================================
# Encoder
# ============================================================
class ECGEncoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=512):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        embedding = self.network(x)
        return embedding


# ============================================================
# Decoder
# ============================================================
class ECGDecoder(nn.Module):
    def __init__(self, embedding_dim, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Linear(64, input_dim)
        )

    def forward(self, embedding):
        reconstruction = self.network(embedding)
        return reconstruction

In [12]:
import torch
import torch.nn as nn

# Number of ECG features
input_dim = X_train.shape[1]
num_classes = np.max(y_train_encoded) + 1

print("Input Dimension:", input_dim)
print("Number of Classes:", num_classes)

class SupervisedAutoencoder(nn.Module):
    def __init__(self, input_dim, num_classes, embedding_dim=512):
        super().__init__()

        # ---------------- Encoder ----------------
        self.encoder = ECGEncoder(
            input_dim=input_dim,
            embedding_dim=embedding_dim
        )

        # ---------------- Decoder ----------------
        self.decoder = ECGDecoder(
            embedding_dim=embedding_dim,
            input_dim=input_dim
        )

        # ---------------- Classification Head ----------------
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # Generate 512-dimensional embedding
        embedding = self.encoder(x)

        # Disease prediction
        logits = self.classifier(embedding)

        # ECG reconstruction
        reconstruction = self.decoder(embedding)

        return logits, reconstruction, embedding

Input Dimension: 16
Number of Classes: 51


In [13]:
# ============================================================
# Required Imports
# ============================================================

import torch
import torch.nn as nn
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# Input Dimension
# ============================================================

input_dim = X_train.shape[1]

# ============================================================
# Get Classes Directly From Training Labels
# ============================================================

unique_classes = np.unique(y_train_encoded)

# IMPORTANT:
# Output layer size must be max_label + 1
num_classes = np.max(y_train_encoded) + 1

print("Input Dimension :", input_dim)
print("Classes Present :", unique_classes)
print("Maximum Label :", np.max(y_train_encoded))
print("Number of Output Classes :", num_classes)

# ============================================================
# Compute Class Weights
# ============================================================

# Compute weights for classes present in training data
weights_present = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=y_train_encoded
)

# Create weight vector for ALL output classes
weights = np.ones(num_classes, dtype=np.float32)

# Fill weights for classes that exist
weights[unique_classes] = weights_present

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=device
)

print("Class Weights Shape:", class_weights.shape)
print("Class Weights:", class_weights)

# ============================================================
# Initialize Model
# ============================================================

model = SupervisedAutoencoder(
    input_dim=input_dim,
    num_classes=num_classes,
    embedding_dim=512
).to(device)

print("Model initialized successfully!")

# ============================================================
# Loss Functions
# ============================================================

classification_criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

reconstruction_criterion = nn.MSELoss()

# Weight of reconstruction loss
alpha = 0.05

# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

# ============================================================
# Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.5
)

print("\nEverything initialized successfully.")

Using device: cpu
Input Dimension : 16
Classes Present : [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 41 42 43 44 45 46 47 48
 49 50]
Maximum Label : 50
Number of Output Classes : 51
Class Weights Shape: torch.Size([51])
Class Weights: tensor([2.4556e+00, 1.5593e+02, 1.5593e+02, 1.1138e+01, 2.0791e+01, 3.3898e+00,
        2.7356e+00, 3.1186e-01, 1.3860e+00, 1.1382e+00, 2.8351e+01, 1.4850e+01,
        1.4175e+01, 1.5066e+00, 2.6097e-01, 1.1550e+01, 2.9421e+00, 1.0395e+02,
        3.1186e+02, 1.0395e+02, 9.7456e+00, 1.5593e+01, 5.0219e-01, 1.5830e+00,
        2.3989e+01, 3.7126e+00, 7.4252e+00, 4.8728e+00, 1.1138e+01, 1.6075e+00,
        1.3271e+00, 4.5527e-01, 5.9973e+00, 6.1149e+00, 5.6702e+00, 1.5593e+01,
        4.4679e-01, 2.6746e-01, 4.4167e-02, 8.6149e-01, 1.0000e+00, 1.8674e+00,
        4.4551e+01, 3.1186e+02, 1.6949e+00, 1.3559e+01, 5.1977e+01, 4.4551e+01,
        1.0395e+02, 1.5593e+02, 8.2068e+00])
Mo

In [14]:
print("Unique labels:")
print(np.unique(y_train_encoded))

print("Maximum label:", np.max(y_train_encoded))
print("Minimum label:", np.min(y_train_encoded))

Unique labels:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 41 42 43 44 45 46 47 48
 49 50]
Maximum label: 50
Minimum label: 0


In [15]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [16]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

In [17]:
# ==========================================
# Training Loop
# ==========================================

num_epochs = 30

train_losses = []
val_losses = []

train_accs = []
val_accs = []

for epoch in range(num_epochs):

    ####################################################
    # TRAINING
    ####################################################
    model.train()

    running_loss = 0.0
    running_cls_loss = 0.0
    running_recon_loss = 0.0

    correct = 0
    total = 0

    for features, labels in train_loader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits, reconstruction, embedding = model(features)

        classification_loss = classification_criterion(
            logits,
            labels
        )

        reconstruction_loss = reconstruction_criterion(
            reconstruction,
            features
        )

        loss = classification_loss + alpha * reconstruction_loss

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        running_cls_loss += classification_loss.item()

        running_recon_loss += reconstruction_loss.item()

        _, predicted = torch.max(logits, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)

    train_cls_loss = running_cls_loss / len(train_loader)

    train_recon_loss = running_recon_loss / len(train_loader)

    train_acc = 100 * correct / total

    train_losses.append(train_loss)

    train_accs.append(train_acc)

    ####################################################
    # VALIDATION
    ####################################################

    model.eval()

    running_val_loss = 0.0

    correct = 0
    total = 0

    with torch.no_grad():

        for features, labels in test_loader:

            features = features.to(device)
            labels = labels.to(device)

            logits, reconstruction, embedding = model(features)

            classification_loss = classification_criterion(
                logits,
                labels
            )

            reconstruction_loss = reconstruction_criterion(
                reconstruction,
                features
            )

            loss = classification_loss + alpha * reconstruction_loss

            running_val_loss += loss.item()

            _, predicted = torch.max(logits, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / len(test_loader)

    val_acc = 100 * correct / total

    val_losses.append(val_loss)

    val_accs.append(val_acc)

    scheduler.step(val_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Cls Loss: {train_cls_loss:.4f} "
        f"| Recon Loss: {train_recon_loss:.4f} "
        f"| Train Acc: {train_acc:.2f}% "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Acc: {val_acc:.2f}%"
    )

Epoch [1/30] | Train Loss: 3.9651 | Cls Loss: 3.9268 | Recon Loss: 0.7662 | Train Acc: 8.23% | Val Loss: 3.9435 | Val Acc: 13.39%
Epoch [2/30] | Train Loss: 3.9185 | Cls Loss: 3.8957 | Recon Loss: 0.4562 | Train Acc: 9.52% | Val Loss: 3.9130 | Val Acc: 22.26%
Epoch [3/30] | Train Loss: 3.8808 | Cls Loss: 3.8621 | Recon Loss: 0.3749 | Train Acc: 10.11% | Val Loss: 3.8719 | Val Acc: 23.13%
Epoch [4/30] | Train Loss: 3.8328 | Cls Loss: 3.8160 | Recon Loss: 0.3359 | Train Acc: 12.79% | Val Loss: 3.8197 | Val Acc: 22.88%
Epoch [5/30] | Train Loss: 3.7741 | Cls Loss: 3.7586 | Recon Loss: 0.3110 | Train Acc: 14.60% | Val Loss: 3.7673 | Val Acc: 25.21%
Epoch [6/30] | Train Loss: 3.7258 | Cls Loss: 3.7112 | Recon Loss: 0.2917 | Train Acc: 18.16% | Val Loss: 3.7235 | Val Acc: 30.49%
Epoch [7/30] | Train Loss: 3.6824 | Cls Loss: 3.6685 | Recon Loss: 0.2789 | Train Acc: 22.02% | Val Loss: 3.6974 | Val Acc: 34.27%
Epoch [8/30] | Train Loss: 3.6599 | Cls Loss: 3.6463 | Recon Loss: 0.2714 | Train Acc

In [18]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

model.eval()

all_labels = []
all_predictions = []

test_loss = 0.0
test_cls_loss = 0.0
test_recon_loss = 0.0

with torch.no_grad():

    for features, labels in test_loader:

        features = features.to(device)
        labels = labels.to(device)

        # Forward pass
        logits, reconstruction, embedding = model(features)

        # Classification loss
        classification_loss = classification_criterion(
            logits,
            labels
        )

        # Reconstruction loss
        reconstruction_loss = reconstruction_criterion(
            reconstruction,
            features
        )

        # Total loss
        loss = classification_loss + alpha * reconstruction_loss

        test_loss += loss.item()
        test_cls_loss += classification_loss.item()
        test_recon_loss += reconstruction_loss.item()

        predictions = torch.argmax(logits, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())


# ==========================================
# Average Losses
# ==========================================

test_loss /= len(test_loader)
test_cls_loss /= len(test_loader)
test_recon_loss /= len(test_loader)


# ==========================================
# Metrics
# ==========================================

accuracy = accuracy_score(all_labels, all_predictions)

precision = precision_score(
    all_labels,
    all_predictions,
    average='weighted',
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average='weighted',
    zero_division=0
)


print("=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"Total Loss          : {test_loss:.4f}")
print(f"Classification Loss : {test_cls_loss:.4f}")
print(f"Reconstruction Loss : {test_recon_loss:.4f}")

print()

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print()

print("Classification Report")
print(classification_report(
    all_labels,
    all_predictions,
    zero_division=0
))

print()

print("Confusion Matrix")
print(confusion_matrix(
    all_labels,
    all_predictions
))

TEST RESULTS
Total Loss          : 3.4737
Classification Loss : 3.4685
Reconstruction Loss : 0.1048

Accuracy  : 0.2957
Precision : 0.4292
Recall    : 0.2957
F1 Score  : 0.3267

Classification Report
              precision    recall  f1-score   support

           0       0.04      0.24      0.07        37
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         1
           3       0.08      0.56      0.14        16
           4       0.09      0.38      0.14         8
           5       0.00      0.00      0.00        26
           6       0.00      0.00      0.00        41
           7       0.00      0.00      0.00       252
           8       0.23      0.22      0.22        55
           9       0.25      0.01      0.03        71
          10       0.00      0.00      0.00         2
          11       0.01      0.14      0.02         7
          12       0.00      0.00      0.00         7
          13       0.10      0.29      0.14

In [19]:
# ============================================================
# Save Complete Supervised Autoencoder
# ============================================================

torch.save(
    model.state_dict(),
    "supervised_autoencoder.pth"
)

print("Complete Supervised Autoencoder saved successfully.")


# ============================================================
# Save Only Encoder
# ============================================================

torch.save(
    model.encoder.state_dict(),
    "ecg_encoder_512.pth"
)

print("Encoder saved successfully.")


# ============================================================
# Save Decoder (Optional)
# ============================================================

torch.save(
    model.decoder.state_dict(),
    "ecg_decoder.pth"
)

print("Decoder saved successfully.")

Complete Supervised Autoencoder saved successfully.
Encoder saved successfully.
Decoder saved successfully.


In [20]:
# ============================================================
# Save Complete Supervised Autoencoder
# ============================================================

torch.save(
    model.state_dict(),
    "supervised_autoencoder.pth"
)

print("Complete Supervised Autoencoder saved successfully.")


# ============================================================
# Save Only Encoder
# ============================================================

torch.save(
    model.encoder.state_dict(),
    "ecg_encoder_512.pth"
)

print("Encoder saved successfully.")


# ============================================================
# Save Decoder (Optional)
# ============================================================

torch.save(
    model.decoder.state_dict(),
    "ecg_decoder.pth"
)

print("Decoder saved successfully.")

Complete Supervised Autoencoder saved successfully.
Encoder saved successfully.
Decoder saved successfully.


In [21]:
# ============================================================
# Generate 512-Dimensional ECG Embeddings
# ============================================================

model.eval()

train_embeddings = []
train_labels = []

with torch.no_grad():

    for features, labels in train_loader:

        features = features.to(device)

        embedding = model.encoder(features)

        train_embeddings.append(
            embedding.cpu().numpy()
        )

        train_labels.append(
            labels.numpy()
        )

train_embeddings = np.vstack(train_embeddings)
train_labels = np.concatenate(train_labels)

print("Training Embeddings Shape :", train_embeddings.shape)

Training Embeddings Shape : (15593, 512)


In [22]:
# ============================================================
# Generate Test Embeddings
# ============================================================

model.eval()

test_embeddings = []
test_labels = []

with torch.no_grad():

    for features, labels in test_loader:

        features = features.to(device)

        embedding = model.encoder(features)

        test_embeddings.append(
            embedding.cpu().numpy()
        )

        test_labels.append(
            labels.numpy()
        )

test_embeddings = np.vstack(test_embeddings)
test_labels = np.concatenate(test_labels)

print("Test Embeddings Shape :", test_embeddings.shape)

Test Embeddings Shape : (3899, 512)


In [23]:
# ============================================================
# Save Embeddings
# ============================================================

np.save(
    "train_embeddings.npy",
    train_embeddings
)

np.save(
    "test_embeddings.npy",
    test_embeddings
)

np.save(
    "train_labels.npy",
    train_labels
)

np.save(
    "test_labels.npy",
    test_labels
)

print("Embeddings saved successfully.")

Embeddings saved successfully.
